# Riproduzione di Baldi, Sadowski & Whiteson (2014)

*Searching for Exotic Particles in High-Energy Physics with Deep Learning*, arXiv:1402.4735

---

## Il problema

Ai collisori adronici la scoperta di nuove particelle e' un problema di classificazione
segnale/fondo. Il benchmark HIGGS contrappone due processi che producono lo **stesso
stato finale** $W^\mp W^\pm b\bar{b}$ ma con cinematica diversa:

- **segnale**: $gg \to H^0 \to W^\mp H^\pm \to W^\mp W^\pm h^0 \to W^\mp W^\pm b\bar{b}$,
  con $m_{H^0} = 425$ GeV e $m_{H^\pm} = 325$ GeV;
- **fondo**: produzione di coppie $t\bar{t}$, con ciascun top che decade in $Wb$.

Poiche' i prodotti osservabili sono identici, la separazione e' possibile solo
sfruttando differenze cinematiche sottili. Gli 11 milioni di eventi sono simulati
con MadGraph5 (generazione), Pythia (shower e adronizzazione) e Delphes (risposta
del rivelatore), a 8 TeV.

## Le due famiglie di variabili

| | cosa sono | quante |
|---|---|---|
| **low-level** | misure dirette del rivelatore: $p_T$, $\eta$, $\phi$ del leptone e dei 4 jet, b-tagging, energia trasversa mancante | 21 |
| **high-level** | masse invarianti ricostruite a mano: $m_{jj}$, $m_{jjj}$, $m_{\ell\nu}$, $m_{j\ell\nu}$, $m_{bb}$, $m_{Wbb}$, $m_{WWbb}$ | 7 |

Le high-level sono funzioni non lineari delle low-level, costruite dai fisici per
identificare gli stati intermedi risonanti. Nel segnale, $m_{Wbb}$ e $m_{WWbb}$ hanno
un picco alle masse ipotizzate di $H^\pm$ e $H^0$; nel fondo, $m_{jjj}$ e $m_{j\ell\nu}$
hanno un picco alla massa del top.

## Cosa hanno fatto gli autori

Tre tipi di classificatore - boosted decision tree, rete shallow (1 strato nascosto)
e rete profonda - su ciascuno dei tre insiemi di input.

Configurazione della rete profonda:

- 300 unita' per strato nascosto, attivazione `tanh`
- inizializzazione normale, $\sigma$ = 0.1 (primo strato), 0.05 (intermedi), 0.001 (uscita)
- SGD, learning rate iniziale 0.05 con decadimento esponenziale **a ogni batch**
- momentum in rampa lineare da 0.9 a 0.99 sulle prime 200 epoche
- weight decay $10^{-5}$, mini-batch da 100 eventi
- early stopping sulla perdita di validation
- AUC su 500.000 eventi di test, media su 5 inizializzazioni casuali

> **Nota sul numero di strati.** Il paper parla di "rete a cinque strati", ma dichiara
> anche 279.901 parametri. Il conteggio torna solo con **4 strati nascosti piu' lo
> strato di uscita**: con 5 nascosti farebbero 370.201. Quindi "cinque strati" include
> l'uscita. La verifica numerica e' nella sezione 2.

## Il risultato centrale

| Tecnica | Low-level | High-level | Complete |
|---|---|---|---|
| BDT | 0.73 | 0.78 | 0.81 |
| NN shallow | 0.733 | 0.777 | 0.816 |
| DN profonda | **0.880** | 0.800 | **0.885** |

Da leggere per righe e per colonne.

**Sulla riga della rete shallow**: passare dalle low-level alle high-level fa salire
l'AUC da 0.733 a 0.777. La rete shallow non riesce a ricostruire le masse invarianti
da sola, e per questo la costruzione manuale delle feature le e' indispensabile.

**Sulla riga della rete profonda**: le sole low-level danno 0.880, praticamente quanto
il set completo (0.885). La rete profonda **scopre da sola** l'informazione contenuta
nelle masse invarianti. In piu', 0.880 supera nettamente lo 0.800 ottenuto sulle sole
high-level: la rete trova potere discriminante *oltre* quello che i fisici avevano
codificato a mano.

## Cosa riproduciamo qui

La griglia completa `{shallow, deep, BDT}` x `{low, high, complete}` sul benchmark
HIGGS, con curve ROC e AUC sul test set canonico (ultimi 500.000 eventi).

**Deviazioni consapevoli dal paper**, discusse in relazione:

- le statistiche di standardizzazione sono calcolate solo sul training set (il paper
  le calcola su train e test insieme, cosa che oggi e' considerata data leakage);
- il BDT usa scikit-learn invece di TMVA;
- tetto di 300 epoche, mentre il paper riporta 200-1000 epoche;
- il pre-training con autoencoder non e' riprodotto, dato che il paper stesso riporta
  che non migliorava le prestazioni.

## Come e' organizzato questo notebook

| sezione | quando eseguirla |
|---|---|
| 1. Verifica dei dati | dopo `prepare_data.py` |
| 2. Verifica dell'architettura | prima del training |
| 3. Prova di addestramento | prima del training vero |
| 4. Risultati | **dopo** che `esperimenti.py` e `bdt.py` hanno girato |
| 5. Figura 8 | dopo la sezione 4 |

L'addestramento vero **non** avviene qui: gira via `src/esperimenti.py` e
`src/bdt.py`, che salvano tutto in `results/`.

---
# 0. Preparazione dell'ambiente

Le due righe magiche ricaricano automaticamente i moduli di `src/` quando li
modifichi, senza dover riavviare il kernel. `sys.path.append` serve perche' il
notebook sta in `notebooks/` mentre i moduli stanno in `src/`.

In [3]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append("../src")

import numpy as np
import torch
import matplotlib.pyplot as plt

PROGETTO = Path("..").resolve()
DATI = PROGETTO / "data" / "processed"
DATI_PICCOLI = PROGETTO / "data" / "processed_small"
RISULTATI = PROGETTO / "results"

print("progetto:", PROGETTO)
print("torch:", torch.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
progetto: /home/alice/Deep-Learning
torch: 2.13.0+cpu


---
# 1. Verifica dei dati

Da eseguire una volta sola, dopo la conversione fatta da `src/prepare_data.py`.
Serve a confermare che gli array siano stati scritti per intero e nel formato giusto.

### 1.1 Integrita' degli array

Quattro controlli. L'ultimo e' il piu' importante: gli array vengono creati vuoti e
riempiti a blocchi, quindi se la conversione si fosse interrotta a meta' la coda
conterrebbe valori casuali senza che nulla segnali l'errore.

In [4]:
X = np.load(DATI / "X.npy", mmap_mode="r")
y = np.load(DATI / "y.npy", mmap_mode="r")

print("forme      :", X.shape, y.shape)      # (11000000, 28) e (11000000, 1)
print("tipo       :", X.dtype)               # float32
print("segnale    :", float(y.mean()))       # circa 0.53
print("NaN        :", bool(np.isnan(X[:100_000]).any()))   # False
print("ultima riga:", X[-1][:6])             # non devono essere zeri o valori assurdi

FileNotFoundError: [Errno 2] No such file or directory: '/home/alice/Deep-Learning/data/processed/X.npy'

### 1.2 I dati sono gia' normalizzati dagli autori

Controllo delle medie di alcune colonne sul dataset grezzo. Le colonne 0, 3 e 5 sono
`lepton_pT`, `missing_energy_magnitude` e `jet1_pt`, tutte grandezze definite positive;
le colonne 1, 2 e 4 sono angolari.

Se le prime danno circa **1** e le seconde circa **0**, significa che lo UCI distribuisce
il dataset **gia' trasformato** con la regola dei Methods del paper. E' esattamente il
caso, e ha due conseguenze da dichiarare in relazione:

1. la nostra standardizzazione in `data.py` e' quasi un'operazione a vuoto (corretta e
   necessaria per generalita', ma con poco da fare su questi dati);
2. il data leakage non e' del tutto eliminabile a valle, perche' la normalizzazione
   originale e' stata calcolata su tutti gli 11 milioni di eventi.

In [ ]:
from features import TUTTE

for i in [0, 1, 2, 3, 4, 5]:
    media = float(X[:200_000, i].mean())
    print(f"colonna {i:2d}  {TUTTE[i]:26s}  media = {media:7.3f}")

---
# 2. Verifica dell'architettura

Controlli da fare **prima** di lanciare ore di addestramento. Verificano che la rete
costruita corrisponda a quella del paper e che sia in condizioni di poter imparare.

### 2.1 Conteggio dei parametri

E' la verifica che ha risolto l'ambiguita' sul numero di strati. La regola, per ogni
strato, e'

$$\text{parametri} = (\text{ingressi} \times \text{uscite}) + \text{uscite}$$

dove il primo termine sono i pesi e il secondo i bias. Per la rete profonda su 28 input:

| strato | ingressi | uscite | parametri |
|---|---|---|---|
| 1 nascosto | 28 | 300 | 8.700 |
| 2 nascosto | 300 | 300 | 90.300 |
| 3 nascosto | 300 | 300 | 90.300 |
| 4 nascosto | 300 | 300 | 90.300 |
| uscita | 300 | 1 | 301 |
| | | **totale** | **279.901** |

Il paper dichiara 279.901 per la rete profonda e 300.001 per la shallow piu' grande
(10.000 unita' nascoste). Se entrambi tornano, la nostra architettura e' la loro.

In [ ]:
from models import rete_profonda, rete_shallow, conta_parametri

profonda = rete_profonda(n_input=28)
shallow_grande = rete_shallow(n_input=28, n_unita=10_000)

print(f"rete profonda, 28 input  : {conta_parametri(profonda):>8,}   (paper: 279.901)")
print(f"rete shallow, 10k unita' : {conta_parametri(shallow_grande):>8,}   (paper: 300.001)")

### 2.2 Inizializzazione dei pesi

Le tre deviazioni standard del paper - 0.1 / 0.05 / 0.001 - non sono arbitrarie.
La `tanh` e' piatta per argomenti grandi: li' la derivata e' quasi nulla e,
moltiplicata strato dopo strato durante la backpropagation, azzera il gradiente.
E' il problema di diffusione del gradiente citato nell'introduzione del paper.

- **0.1** al primo strato, che riceve dati standardizzati;
- **0.05** agli strati interni, che ricevono uscite di `tanh` gia' limitate in $[-1,1]$.
  Da notare: $1/\sqrt{300} \approx 0.058$, cioe' gli autori hanno scelto per tentativi
  un valore quasi coincidente con l'inizializzazione di Xavier;
- **0.001** all'uscita, cosi' la rete parte prevedendo probabilita' vicine a 0.5 per
  tutti gli eventi: nessun pregiudizio iniziale.

I valori misurati non coincidono esattamente con quelli nominali perche' sono
**deviazioni standard campionarie**: l'oscillazione attesa e' circa $1/\sqrt{2N}$,
quindi trascurabile sugli strati da 90.000 pesi e visibile sull'uscita che ne ha 300.

In [ ]:
modello = rete_profonda(n_input=21)

for i, strato in enumerate(modello.nascosti):
    atteso = 0.1 if i == 0 else 0.05
    print(f"nascosto {i}: std misurata = {strato.weight.std().item():.4f}   (attesa {atteso})")
print(f"uscita    : std misurata = {modello.uscita.weight.std().item():.5f}   (attesa 0.001)")

Conseguenza dello 0.001 sull'uscita: la rete appena inizializzata non ha opinioni.
Tutte le probabilita' previste devono essere vicinissime a 0.5.

Nota anche che il modello restituisce un **logit**, non una probabilita': la sigmoide
vive dentro `BCEWithLogitsLoss` per stabilita' numerica. Per l'AUC non serve applicarla,
essendo monotona non cambia l'ordinamento degli eventi.

In [ ]:
x_finto = torch.randn(8, 21)
logit = modello(x_finto)

print("forma dell'uscita:", tuple(logit.shape))       # (8, 1)
print("probabilita':", torch.sigmoid(logit).squeeze().detach().numpy().round(4))

### 2.3 Propagazione delle attivazioni

Questo controllo dice se la rete e' **addestrabile**, e rende concreto il problema di
cui il paper parla solo a parole. Va fatto sui dati veri, non su rumore casuale.

Due modi di fallire:

- se le attivazioni medie **crollano** verso zero strato dopo strato, il segnale si
  spegne e i gradienti negli strati profondi saranno minuscoli;
- se la frazione satura **sale** verso 1, le `tanh` sono finite nella zona piatta.

Cosa vogliamo vedere: attenuazione dolce e saturazione trascurabile.

Questa cella e' anche il materiale per il confronto `tanh` contro `ReLU`
nell'estensione sullo stack moderno: gli stessi numeri, affiancati, mostrano
concretamente da dove viene la differenza.

In [ ]:
from data import prepara_dati

# dataset piccolo: qui serve solo un campione rappresentativo, non tutti i dati
dati_prova = prepara_dati(
    feature_set="low",
    cartella=DATI_PICCOLI,
    n_train=60_000, n_val=20_000, n_test=20_000,
)

X_prova, y_prova = dati_prova["train"]
x = torch.from_numpy(X_prova[:1000])

logit, attivazioni = modello(x, restituisci_attivazioni=True)

for i, a in enumerate(attivazioni):
    print(f"strato {i}: |attivazione| media = {a.abs().mean():.4f}, "
          f"frazione satura (>0.9) = {(a.abs() > 0.9).float().mean():.3f}")

### 2.4 Propagazione dei gradienti

Il complementare del controllo precedente: le attivazioni raccontano cosa succede in
avanti, i gradienti cosa succede all'indietro.

Le norme non devono differire di ordini di grandezza tra il primo e l'ultimo strato.
Un gradiente di $10^{-6}$ sul primo strato e $10^{-1}$ sull'ultimo sarebbe la firma
del problema di diffusione del gradiente.

In [ ]:
perdita = torch.nn.BCEWithLogitsLoss()(logit, torch.from_numpy(y_prova[:1000]))
perdita.backward()

for i, strato in enumerate(modello.nascosti):
    print(f"strato {i}: norma del gradiente = {strato.weight.grad.norm():.6f}")
print(f"uscita   : norma del gradiente = {modello.uscita.weight.grad.norm():.6f}")

---
# 3. Prova di addestramento

Verifica end-to-end su dati ridotti, per confermare che la catena funzioni prima di
lanciare le run vere. **Non guardare l'AUC che esce**: 60.000 eventi e 15 epoche non
bastano per nulla.

Quello che deve succedere:

- la perdita deve **scendere** partendo da circa 0.693 (che e' $\ln 2$, il valore di
  una rete che tira a indovinare);
- l'AUC deve salire sopra 0.5;
- il momentum deve salire da 0.900 a 0.990 e poi restare fisso.

Il batch e' 1000 invece di 100 perche' qui interessa solo la velocita'.

In [ ]:
from train import addestra, valuta

modello_prova = rete_profonda(n_input=21)

storia_prova = addestra(
    modello_prova, dati_prova,
    batch=1000,
    epoche_rampa=5,
    max_epoche=15,
    pazienza=3,
    seme=0,
)

Le curve di apprendimento della prova. Se le due perdite (train e validation) restano
vicine, non c'e' overfitting; se divergono, la rete sta memorizzando il training set.

In [ ]:
from evaluate import disegna_storia

disegna_storia(storia_prova, titolo="Prova su dataset ridotto")
plt.show()

---
# 4. Risultati

**Da eseguire dopo** che `src/esperimenti.py` e `src/bdt.py` hanno completato tutte
le configurazioni. Le celle leggono i file salvati in `results/` e non riaddestrano
nulla.

Configurazioni attese: `{shallow, deep, bdt}` x `{low, high, complete}`, 5 semi ciascuna.

### 4.1 Cosa c'e' in results/

In [ ]:
riepiloghi = sorted(RISULTATI.glob("*_riepilogo.json"))

print(f"{len(riepiloghi)} configurazioni completate:\n")
for f in riepiloghi:
    print("  ", f.name)

### 4.2 Tabella I

La tabella dei risultati, nello stesso formato del paper: AUC medio sui semi, con
deviazione standard fra parentesi.

Il criterio di successo **non e' la coincidenza decimale** con i valori originali, ma
la conservazione di tre relazioni:

1. shallow low **nettamente sotto** shallow high - la rete piccola non ricostruisce
   le masse invarianti;
2. deep low **circa uguale** a deep complete - la rete profonda le ricostruisce da sola;
3. deep low **nettamente sopra** deep high - e trova anche altro.

Se queste tre disuguaglianze reggono con valori assoluti leggermente diversi, il paper
e' riprodotto. Se i valori coincidessero ma le disuguaglianze fossero rotte, no.

In [ ]:
import json

# valori del paper, per il confronto
PAPER = {
    ("bdt", "low"): 0.73,     ("bdt", "high"): 0.78,     ("bdt", "complete"): 0.81,
    ("shallow", "low"): 0.733, ("shallow", "high"): 0.777, ("shallow", "complete"): 0.816,
    ("deep", "low"): 0.880,   ("deep", "high"): 0.800,   ("deep", "complete"): 0.885,
}

risultati = {}
for f in RISULTATI.glob("*_riepilogo.json"):
    with open(f) as fh:
        info = json.load(fh)
    risultati[(info["modello"], info["feature_set"])] = info

print(f"{'':10s} {'low-level':>22s} {'high-level':>22s} {'complete':>22s}")
print("-" * 80)

for modello in ["bdt", "shallow", "deep"]:
    riga = f"{modello:10s}"
    for fs in ["low", "high", "complete"]:
        info = risultati.get((modello, fs))
        atteso = PAPER[(modello, fs)]
        if info is None:
            riga += f"{'-':>12s} ({atteso:.3f})"
        else:
            riga += f"  {info['auc_medio']:.4f} ({info['auc_dev_std']:.4f})"
    print(riga)

print()
print("Valori del paper, per confronto:")
print(f"{'':10s} {'low':>10s} {'high':>10s} {'complete':>10s}")
for modello in ["bdt", "shallow", "deep"]:
    riga = f"{modello:10s}"
    for fs in ["low", "high", "complete"]:
        riga += f"{PAPER[(modello, fs)]:>10.3f}"
    print(riga)

### 4.3 Le tre disuguaglianze

Verifica esplicita del criterio di successo.

In [ ]:
def auc(modello, fs):
    info = risultati.get((modello, fs))
    return info["auc_medio"] if info else None

controlli = [
    ("shallow low  <  shallow high", auc("shallow", "low"), auc("shallow", "high"), "<"),
    ("deep low     ~= deep complete", auc("deep", "low"), auc("deep", "complete"), "~"),
    ("deep low     >  deep high", auc("deep", "low"), auc("deep", "high"), ">"),
]

for nome, a, b, segno in controlli:
    if a is None or b is None:
        print(f"  {nome:32s}  dati mancanti")
        continue
    if segno == "<":
        esito = a < b
    elif segno == ">":
        esito = a > b
    else:
        esito = abs(a - b) < 0.02
    simbolo = "OK  " if esito else "NO  "
    print(f"  {simbolo}{nome:32s}  {a:.4f}  vs  {b:.4f}   (differenza {b - a:+.4f})")

### 4.4 Figura 7

Le curve di prestazione, nella convenzione della fisica delle alte energie.
**Non e' una ROC standard**: in ascissa la signal efficiency (TPR, quanti eventi di
segnale vengono tenuti), in ordinata la background rejection (1-FPR, quanto fondo
viene scartato). E' la stessa informazione di una ROC con l'asse verticale ribaltato,
e la curva va dall'angolo in alto a sinistra a quello in basso a destra.

Il pannello (a) e' il metodo tradizionale, il (b) il deep learning.

In [ ]:
from evaluate import carica_curva

def curve_di(modello, seme=0):
    "Carica le tre curve di un modello, una per feature set."
    curve = []
    for fs in ["low", "high", "complete"]:
        percorso = RISULTATI / f"{modello}_{fs}_seme{seme}_roc.npz"
        if percorso.exists():
            c = carica_curva(percorso)
            c["nome"] = f"{fs}-level"
            curve.append(c)
    return curve

fig, assi = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, modello, titolo in zip(assi, ["shallow", "deep"],
                               ["(a) rete shallow", "(b) rete profonda"]):
    for c in curve_di(modello):
        ax.plot(c["signal_efficiency"], c["background_rejection"],
                label=f"{c['nome']}  (AUC = {c['auc']:.3f})", linewidth=1.6)
    ax.set_xlabel("Signal efficiency")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend(loc="lower left", fontsize=9)
    ax.set_title(titolo)

assi[0].set_ylabel("Background rejection")
fig.tight_layout()
fig.savefig(RISULTATI / "figura7.png", dpi=150)
plt.show()

### 4.5 Curve di apprendimento

Serve a rispondere a una domanda precisa: **il tetto di 300 epoche ha penalizzato
qualche configurazione?**

Se l'AUC di validation era ancora in salita all'ultima epoca, il limite era il budget
di calcolo e non il metodo. E' un'informazione onesta da riportare in relazione, e
questo grafico la documenta meglio di qualunque giustificazione a parole.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for modello in ["shallow", "deep"]:
    for fs in ["low", "high", "complete"]:
        percorso = RISULTATI / f"{modello}_{fs}_seme0_storia.json"
        if not percorso.exists():
            continue
        with open(percorso) as f:
            storia = json.load(f)
        stile = "-" if modello == "deep" else "--"
        ax.plot(range(1, len(storia["auc_val"]) + 1), storia["auc_val"],
                stile, label=f"{modello} {fs}", linewidth=1.3)

ax.set_xlabel("epoca")
ax.set_ylabel("AUC su validation")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
ax.set_title("Convergenza: le curve sono arrivate in piano?")
fig.tight_layout()
plt.show()

---
# 5. Figura 8: perche' la rete profonda vince

Questa e' la figura piu' interessante del paper, e il precursore concettuale del
linear probing.

**L'idea.** Si prendono due classificatori con prestazioni confrontabili - la rete
profonda su 21 feature low-level e la rete shallow su 7 high-level - e si fissa per
entrambi una soglia che scarti il **90% del fondo**. Poi si guarda *quali* eventi
ciascuno seleziona, in termini di $m_{Wbb}$ e $m_{WWbb}$, cioe' proprio le masse
invarianti che codificano le risonanze $H^\pm$ e $H^0$.

**Cosa trova il paper.** La rete shallow seleziona eventi vicini al picco di segnale
e si tiene lontana dalla regione dominata dal fondo: sta usando le masse invarianti
esattamente come le userebbe un fisico. La rete profonda seleziona gli stessi eventi
vicini al picco, **ma in piu' recupera eventi nella regione dominata dal fondo**.

**Perche' conta.** Significa che la rete profonda ha ricostruito da sola la regione
ricca di segnale identificata dalle variabili fisiche, e in aggiunta ha trovato modi
di penetrare nella zona dove le variabili fisiche non discriminano. E' la traduzione
visiva del "trova potere discriminante oltre le high-level".

### 5.1 Caricare i modelli addestrati

I file `.pt` contengono solo i pesi, non l'architettura: bisogna ricostruire la rete
con gli stessi parametri e poi caricarci dentro i pesi salvati.

In [ ]:
from features import INDICI, TUTTE

def carica_modello(tipo, feature_set, seme=0):
    "Ricostruisce l'architettura e vi carica i pesi salvati."
    n_input = len(INDICI[feature_set])
    if tipo == "deep":
        modello = rete_profonda(n_input=n_input)
    else:
        modello = rete_shallow(n_input=n_input)

    percorso = RISULTATI / f"{tipo}_{feature_set}_seme{seme}_modello.pt"
    modello.load_state_dict(torch.load(percorso, map_location="cpu"))
    modello.eval()
    return modello

dn21 = carica_modello("deep", "low")        # rete profonda, 21 input low-level
nn7 = carica_modello("shallow", "high")     # rete shallow, 7 input high-level

print("modelli caricati")

### 5.2 Predizioni e soglie al 90% di reiezione del fondo

La soglia si calcola **sugli eventi di fondo**: e' il valore tale che solo il 10% di
essi lo superi. Poiche' i due modelli hanno scale di uscita diverse, ciascuno ha la
propria soglia; cio' che si tiene fisso e' la reiezione, non il numero.

In [ ]:
from train import predici

# ogni modello vuole il proprio feature set, standardizzato allo stesso modo
dati_low = prepara_dati(feature_set="low", silenzioso=True)
dati_high = prepara_dati(feature_set="high", silenzioso=True)

X_test_low, y_test = dati_low["test"]
X_test_high, _ = dati_high["test"]

p_dn21 = predici(dn21, X_test_low)
p_nn7 = predici(nn7, X_test_high)

etichette = y_test.ravel()
e_fondo = etichette == 0

soglia_dn21 = np.quantile(p_dn21[e_fondo], 0.90)
soglia_nn7 = np.quantile(p_nn7[e_fondo], 0.90)

sel_dn21 = p_dn21 >= soglia_dn21
sel_nn7 = p_nn7 >= soglia_nn7

print(f"DN21: soglia {soglia_dn21:.4f}  ->  efficienza sul segnale "
      f"{sel_dn21[etichette == 1].mean():.3f}")
print(f"NN7 : soglia {soglia_nn7:.4f}  ->  efficienza sul segnale "
      f"{sel_nn7[etichette == 1].mean():.3f}")

### 5.3 Gli istogrammi

Le due masse invarianti si prendono dall'array originale, non da quelli standardizzati:
qui interessa la distribuzione cosi' com'e' nel dataset. `m_wbb` e `m_wwbb` sono le
colonne 26 e 27 di `X`.

Nel grafico, quattro curve per pannello: segnale puro, fondo puro, e gli eventi
selezionati da ciascuno dei due classificatori. Tutte normalizzate ad area unitaria,
per poterle confrontare come forme.

In [ ]:
X_test_completo = np.load(DATI / "X.npy", mmap_mode="r")[-len(y_test):]

i_wbb = TUTTE.index("m_wbb")
i_wwbb = TUTTE.index("m_wwbb")

fig, assi = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, colonna, nome in zip(assi, [i_wbb, i_wwbb], ["$m_{Wbb}$", "$m_{WWbb}$"]):
    valori = np.asarray(X_test_completo[:, colonna])
    bins = np.linspace(np.quantile(valori, 0.01), np.quantile(valori, 0.99), 60)

    ax.hist(valori[etichette == 1], bins=bins, density=True, histtype="step",
            color="black", linewidth=1.6, label="segnale")
    ax.hist(valori[etichette == 0], bins=bins, density=True, histtype="step",
            color="red", linestyle=":", linewidth=1.4, label="fondo")
    ax.hist(valori[sel_dn21], bins=bins, density=True, histtype="step",
            color="tab:blue", linestyle="--", linewidth=1.4, label="DN21 (rej = 0.9)")
    ax.hist(valori[sel_nn7], bins=bins, density=True, histtype="step",
            color="tab:purple", linestyle="-.", linewidth=1.4, label="NN7 (rej = 0.9)")

    ax.set_xlabel(nome + "  (unita' normalizzate)")
    ax.set_ylabel("frazione di eventi")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(RISULTATI / "figura8.png", dpi=150)
plt.show()

**Come leggere il grafico.** Confronta la curva blu (rete profonda) con quella viola
(rete shallow). Se la blu ha piu' peso nelle code, lontano dal picco di segnale,
significa che la rete profonda sta recuperando eventi che le variabili fisiche da sole
scarterebbero. E' la conclusione del paper: la rete profonda ha ritrovato la regione
ricca di segnale identificata dai fisici e ha in piu' trovato modi di intaccare la
regione dominata dal fondo.

Nota che le ascisse sono in **unita' normalizzate**, non in GeV: il dataset pubblicato
e' gia' riscalato e le unita' fisiche originali non sono recuperabili. Se servisse un
riferimento indicativo, si puo' sfruttare il fatto che $m_{\ell\nu}$ ha un picco
stretto a 80,4 GeV per stimare a posteriori un fattore di scala.

---
# 6. Stato della riproduzione

| elemento del paper | stato |
|---|---|
| Tabella I, AUC | sezione 4.2 |
| Figura 7, curve di prestazione | sezione 4.4 |
| Figura 8, distribuzioni selezionate | sezione 5 |
| Tabella I, discovery significance | da fare |
| Tabella Suppl. 3, studio di profondita' | facoltativo |
| Tabella Suppl. 4, regressione delle high-level | sostituita dal linear probing |
| Benchmark SUSY | fuori perimetro |

## Passi successivi: le estensioni

1. **Confronto stack 2014 contro moderno** - `tanh`/SGD contro ReLU/Adam/BatchNorm/dropout.
   Domanda: quanto del divario shallow/deep dipende dall'architettura profonda e quanto
   dalle difficolta' di ottimizzazione dell'epoca? Le celle 2.3 e 2.4 di questo notebook
   danno gia' i numeri da affiancare.

2. **Learning curve** - AUC in funzione di $N_{\text{train}}$ sui tre feature set.
   Ipotesi: la scoperta automatica delle feature richiede dati, quindi a piccolo $N$ le
   high-level restano vantaggiose. Individuare la soglia quantifica il valore della
   conoscenza fisica in termini di eventi simulati equivalenti.

3. **Linear probing** - congelare la rete profonda addestrata sulle low-level, estrarre
   le attivazioni di ogni strato (il gancio e' gia' in `models.py`), e verificare con
   regressioni lineari quanto le 7 masse invarianti siano linearmente decodificabili.
   Verifica **diretta** della tesi che il paper argomenta solo indirettamente, per via
   delle prestazioni.